# MEDISCOPE — 06 Interpretability and Clinical Translation

## Understanding what the models are doing — and what they are not doing

High predictive performance is not sufficient for clinical decision support. This notebook examines model-specific importance signals, compares Logistic Regression and XGBoost outputs, and translates disagreement/error behaviour into the MEDISCOPE workflow.

> **Important:** feature importance and model coefficients are associations learned from the training data. They are **not causal clinical effects**.

## 1. Project setup

In [ ]:
from pathlib import Path
import sys

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

pd.set_option("display.max_columns", 120)
pd.set_option("display.max_rows", 100)
pd.set_option("display.float_format", lambda value: f"{value:,.4f}")


def find_project_root(start: Path | None = None) -> Path:
    """Locate the MEDISCOPE repository root from common notebook launch locations."""
    start = (start or Path.cwd()).resolve()

    for candidate in [start, *start.parents]:
        if (
            (candidate / "src").is_dir()
            and (candidate / "api").is_dir()
            and (candidate / "requirements.txt").exists()
        ):
            return candidate

    raise FileNotFoundError(
        "Unable to locate the MEDISCOPE repository root. "
        "Run this notebook from the repository or notebooks directory."
    )


PROJECT_ROOT = find_project_root()

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

DATA_DIR = PROJECT_ROOT / "data"
RAW_DIR = DATA_DIR / "raw"
PROCESSED_DIR = DATA_DIR / "processed"
MODEL_DIR = PROJECT_ROOT / "models" / "trained"
REPORT_DIR = PROJECT_ROOT / "reports" / "evaluation"

print(f"Project root: {PROJECT_ROOT}")

In [ ]:
import json
import joblib

METADATA_FILE = MODEL_DIR / "training_metadata.json"
TEST_FILE = PROCESSED_DIR / "03_test.parquet"

metadata = json.loads(METADATA_FILE.read_text(encoding="utf-8"))
feature_order = metadata["feature_order"]
target_column = metadata["target_column"]

lr_path = MODEL_DIR / "logistic_regression_pipeline.joblib"
xgb_path = MODEL_DIR / "xgboost_pipeline.joblib"

for required in [TEST_FILE, lr_path, xgb_path]:
    if not required.exists():
        raise FileNotFoundError(f"Required artefact not found: {required}")

test_df = pd.read_parquet(TEST_FILE)
X_test = test_df[feature_order]
y_test = test_df[target_column]

logistic_model = joblib.load(lr_path)
xgboost_model = joblib.load(xgb_path)

## 2. Helper: locate the final estimator in a fitted pipeline

In [ ]:
def final_estimator(model):
    """Return the final fitted estimator from a scikit-learn Pipeline or the model itself."""
    if hasattr(model, "steps") and model.steps:
        return model.steps[-1][1]
    return model


lr_estimator = final_estimator(logistic_model)
xgb_estimator = final_estimator(xgboost_model)

print("Logistic final estimator:", type(lr_estimator).__name__)
print("XGBoost final estimator:", type(xgb_estimator).__name__)

## 3. Logistic Regression coefficients

In [ ]:
if hasattr(lr_estimator, "coef_"):
    coefficients = np.asarray(lr_estimator.coef_).reshape(-1)

    if len(coefficients) == len(feature_order):
        lr_importance = pd.DataFrame({
            "feature": feature_order,
            "coefficient": coefficients,
        })

        lr_importance["absolute_coefficient"] = (
            lr_importance["coefficient"].abs()
        )

        display(
            lr_importance
            .sort_values("absolute_coefficient", ascending=False)
            .head(25)
        )

        top_lr = (
            lr_importance
            .nlargest(20, "absolute_coefficient")
            .sort_values("absolute_coefficient")
        )

        plt.figure(figsize=(10, 7))
        plt.barh(top_lr["feature"], top_lr["absolute_coefficient"])
        plt.xlabel("Absolute fitted coefficient")
        plt.ylabel("Feature")
        plt.title("Logistic Regression — largest absolute coefficients")
        plt.tight_layout()
        plt.show()
    else:
        print(
            "Coefficient count does not directly match the persisted input feature count. "
            "Inspect the fitted preprocessing transformation before assigning names."
        )
else:
    print("The persisted Logistic Regression estimator does not expose coef_.")

### Interpretation caution

For Logistic Regression:

- a positive coefficient generally pushes the log-odds toward the positive/LTFU class;
- a negative coefficient pushes in the opposite direction;
- magnitude is affected by feature scaling and representation;
- correlated predictors can redistribute coefficient magnitude;
- coefficients should not be interpreted as causal effects.

## 4. XGBoost feature importance

In [ ]:
if hasattr(xgb_estimator, "feature_importances_"):
    importances = np.asarray(xgb_estimator.feature_importances_).reshape(-1)

    if len(importances) == len(feature_order):
        xgb_importance = pd.DataFrame({
            "feature": feature_order,
            "importance": importances,
        }).sort_values("importance", ascending=False)

        display(xgb_importance.head(25))

        top_xgb = (
            xgb_importance
            .head(20)
            .sort_values("importance")
        )

        plt.figure(figsize=(10, 7))
        plt.barh(top_xgb["feature"], top_xgb["importance"])
        plt.xlabel("Model feature importance")
        plt.ylabel("Feature")
        plt.title("XGBoost — top model importance values")
        plt.tight_layout()
        plt.show()
    else:
        print(
            "Importance count does not directly match the persisted input feature count. "
            "Inspect the fitted preprocessing transformation before assigning names."
        )
else:
    print("The persisted XGBoost estimator does not expose feature_importances_.")

XGBoost importance is also model-specific. A high importance indicates that a variable was useful to the fitted tree ensemble; it does not prove that changing that variable would cause a patient's clinical outcome to change.

## 5. Compare Logistic Regression and XGBoost probabilities

In [ ]:
lr_probability = logistic_model.predict_proba(X_test)[:, 1]
xgb_probability = xgboost_model.predict_proba(X_test)[:, 1]

probability_comparison = pd.DataFrame({
    "actual_target": y_test.to_numpy(),
    "logistic_probability": lr_probability,
    "xgboost_probability": xgb_probability,
})

probability_comparison["absolute_probability_gap"] = (
    probability_comparison["logistic_probability"]
    - probability_comparison["xgboost_probability"]
).abs()

probability_comparison.describe()

## 6. Model agreement at the operational threshold

In [ ]:
THRESHOLD = 0.50

probability_comparison["logistic_class"] = (
    probability_comparison["logistic_probability"] >= THRESHOLD
).astype(int)

probability_comparison["xgboost_class"] = (
    probability_comparison["xgboost_probability"] >= THRESHOLD
).astype(int)

probability_comparison["models_agree"] = (
    probability_comparison["logistic_class"]
    == probability_comparison["xgboost_class"]
)

agreement_summary = probability_comparison["models_agree"].value_counts()
agreement_rate = probability_comparison["models_agree"].mean()

print(f"Agreement rate: {agreement_rate * 100:.2f}%")
display(agreement_summary.to_frame("records"))

The MEDISCOPE interface preserves disagreement rather than averaging it away. A disagreement can be clinically useful information because it identifies cases where two differently structured models reach different threshold classifications.

## 7. Inspect the largest probability disagreements

In [ ]:
largest_disagreements = (
    probability_comparison
    .sort_values("absolute_probability_gap", ascending=False)
    .head(25)
)

largest_disagreements

These cases are useful candidates for deeper review or future explainability analysis because the two models assign materially different risk probabilities.

## 8. Agreement categories used for decision-support thinking

In [ ]:
def agreement_category(row):
    lr_high = row["logistic_probability"] >= THRESHOLD
    xgb_high = row["xgboost_probability"] >= THRESHOLD

    if lr_high and xgb_high:
        return "BOTH_ABOVE_THRESHOLD"
    if (not lr_high) and (not xgb_high):
        return "BOTH_BELOW_THRESHOLD"
    return "MODEL_DISAGREEMENT"


probability_comparison["agreement_category"] = (
    probability_comparison.apply(agreement_category, axis=1)
)

category_counts = (
    probability_comparison["agreement_category"]
    .value_counts()
)

category_counts.to_frame("records")

This mirrors the broader MEDISCOPE design principle used in clinician intelligence:

- both models above threshold → higher-priority review signal;
- both below threshold → lower model-based risk signal;
- model disagreement → review the context rather than forcing consensus;
- no stored assessment → prediction coverage/data-quality issue rather than a low-risk conclusion.

## 9. False negatives deserve explicit attention

In [ ]:
analysis = probability_comparison.copy()

for model_name in ["logistic", "xgboost"]:
    predicted = analysis[f"{model_name}_class"]

    false_negative_mask = (
        (analysis["actual_target"] == 1)
        & (predicted == 0)
    )

    false_positive_mask = (
        (analysis["actual_target"] == 0)
        & (predicted == 1)
    )

    print(
        f"{model_name.title()}: "
        f"{false_negative_mask.sum():,} false negatives; "
        f"{false_positive_mask.sum():,} false positives"
    )

In a retention-prioritisation context, false negatives are especially important because they represent LTFU cases that would not cross the chosen model threshold.

However, reducing false negatives by lowering the threshold can increase false positives and intervention workload. Threshold choice should therefore reflect service capacity and the relative cost of the two error types.

## 10. Why interpretability does not mean causality

The model learns statistical relationships present in the historical dataset. Several limitations must be kept visible:

1. **Association is not causation.**
2. Geographic or demographic variables can reflect structural differences in service delivery rather than intrinsic patient risk.
3. Missingness may encode both clinical processes and reporting practices.
4. One-hot encoded regimen/geography features should not be interpreted independently of the wider data context.
5. External populations may have different relationships.
6. Predictive importance can change under dataset/model drift.
7. Interpretability tools should support, not replace, domain review.

## 11. Translation into MEDISCOPE

The application operationalises these modelling lessons in several ways:

- probabilities from Logistic Regression and XGBoost remain separate;
- agreement/disagreement is visible;
- prediction history is retained;
- the clinician can inspect longitudinal clinical records;
- a fresh prediction requires an intentional clinician action;
- the output includes a clinical disclaimer;
- predictions are decision-support evidence, not autonomous treatment instructions;
- synthetic demonstration data is separated from the research model-development dataset.

## Key findings

- Logistic Regression offers a comparatively interpretable probabilistic model and achieved the strongest final held-out performance.
- XGBoost provides a complementary non-linear view of the same feature schema.
- Feature importance is useful for understanding the fitted model but must not be treated as causal evidence.
- Model disagreement is preserved as useful uncertainty information.
- False-negative and false-positive trade-offs matter operationally.
- MEDISCOPE embeds model outputs in a human-review workflow rather than automating clinical decisions.

## End of the modelling notebook series

Together, notebooks 01–06 document the analytical path from source-data understanding to responsible model operationalisation.